# MmarakaAI Classical Machine Learning: Stage B and Stage C

This notebook is the primary reproducible benchmarking workflow for classical machine learning experiments in MmarakaAI.

Stage B performs model-based feature selection with Random Forest importance, benchmarks Random Forest and LightGBM across top-N feature subsets, and evaluates the winning baseline on a held-out chronological test set. Stage C preserves that baseline, adds LightGBM-based feature ranking, compares feature-selection pipelines, and performs time-series hyperparameter optimization for LightGBM and Random Forest.


## 1. Environment Setup

The workflow is designed to run locally or in Google Colab. In Colab, missing packages are installed automatically. Locally, the notebook expects dependencies to be installed in the active environment and raises a clear message if they are missing.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

PROJECT_NAME = "MmarakaAI"
REPOSITORY_URL = "https://github.com/Wiz-Tech-Kid/MmarakaAI.git"

try:
    import google.colab  # type: ignore[import-not-found]
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# In Colab, clone the project into /content and make it the working directory.
# Locally, keep the existing repository working directory unchanged.
if IN_COLAB:
    colab_root = Path("/content")
    project_root = colab_root / PROJECT_NAME
    if not project_root.exists():
        subprocess.check_call(["git", "clone", REPOSITORY_URL, str(project_root)])
    os.chdir(project_root)
else:
    current_path = Path.cwd().resolve()
    local_root = next(
        (candidate for candidate in [current_path, *current_path.parents] if (candidate / "data").is_dir() and (candidate / "src").is_dir()),
        None,
    )
    if local_root is not None:
        os.chdir(local_root)

ROOT = Path.cwd().resolve()
print(f"Running in {'Google Colab' if IN_COLAB else 'local VS Code'}")
print(f"Project root: {ROOT}")

if not (ROOT / "data").is_dir() or not (ROOT / "src").is_dir():
    raise FileNotFoundError(
        "The MmarakaAI repository was not found. In Colab, rerun this cell so it can clone the repository."
    )

REQUIREMENTS_PATH = ROOT / "requirements.txt"
if not REQUIREMENTS_PATH.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQUIREMENTS_PATH}")

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "lightgbm": "lightgbm",
    "joblib": "joblib",
    "IPython": "ipython",
}
missing_packages = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages and IN_COLAB:
    print(f"Installing missing packages from {REQUIREMENTS_PATH.name}: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS_PATH)])
elif missing_packages:
    raise ImportError(
        "Missing required packages: "
        + ", ".join(missing_packages)
        + ". Install them locally with: .venv/bin/pip install -r requirements.txt"
    )

print("Repository and dependencies verified")

## 2. Imports, Configuration, and Paths

All randomness is controlled with a fixed seed. Paths are resolved by walking upward from the current working directory until the project data file is found, so the notebook can be launched from the repository root or from the `notebooks/` directory.


In [ ]:
import logging
import math
import time
import warnings
from dataclasses import dataclass
from typing import Callable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_SEED = 42
TARGET_COLUMN = "food_price_inflation"
DATE_COLUMN = "Date"
TEST_SIZE = 0.20
CV_SPLITS = 5
TOP_N_OPTIONS = [25, 50, 100, 200, 500]

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("mmarakai_stage_b")
np.random.seed(RANDOM_SEED)

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        expected = candidate / "data" / "features" / "mmarakai_clean_feature_matrix.csv"
        if expected.exists():
            return candidate
    raise FileNotFoundError("Could not find data/features/mmarakai_clean_feature_matrix.csv from the current path.")

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "features" / "mmarakai_clean_feature_matrix.csv"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "output"
FIGURE_DIR = OUTPUT_DIR / "analysis" / "figures" / "ml"

MODEL_PATH = MODELS_DIR / "stageB_best_classic_ml_model.joblib"
SELECTED_FEATURES_PATH = OUTPUT_DIR / "selected_features.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "feature_importance.csv"
MODEL_COMPARISON_PATH = OUTPUT_DIR / "model_comparison.csv"
FINAL_METRICS_PATH = OUTPUT_DIR / "final_test_metrics.csv"

for directory in [MODELS_DIR, OUTPUT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Input data: %s", DATA_PATH)


## 3. Load and Validate the Stage A Clean Feature Matrix

`Date` is retained for chronological sorting and plotting, but it is not used as a predictor. The target is `food_price_inflation`.


In [ ]:
from pathlib import Path
import logging

import numpy as np
import pandas as pd

if "TARGET_COLUMN" not in globals():
    TARGET_COLUMN = "food_price_inflation"
if "DATE_COLUMN" not in globals():
    DATE_COLUMN = "Date"

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    search_roots = [
        start,
        *start.parents,
        Path("/content") / "MmarakaAI",
        Path.home() / "Documents" / "New Folder" / "MmarakaAI",
    ]
    for root in search_roots:
        expected = root / "data" / "features" / "mmarakai_clean_feature_matrix.csv"
        if expected.exists():
            return root
    raise FileNotFoundError(
        "Could not find data/features/mmarakai_clean_feature_matrix.csv. "
        "Run the environment setup cell first or open the notebook from the project directory."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "features" / "mmarakai_clean_feature_matrix.csv"
if "logger" not in globals():
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
    logger = logging.getLogger("mmarakai_stage_b")

logger.info("Loading Stage A clean feature matrix")
data = pd.read_csv(DATA_PATH)
if DATE_COLUMN not in data.columns:
    raise KeyError(f"Missing required date column: {DATE_COLUMN}")
if TARGET_COLUMN not in data.columns:
    raise KeyError(f"Missing required target column: {TARGET_COLUMN}")

data[DATE_COLUMN] = pd.to_datetime(data[DATE_COLUMN], errors="coerce")
if data[DATE_COLUMN].isna().any():
    raise ValueError("Date column contains unparseable values.")
data = data.sort_values(DATE_COLUMN, kind="mergesort").reset_index(drop=True)

if data[TARGET_COLUMN].isna().any():
    raise ValueError("Target column contains missing values. Resolve target missingness before model training.")
if data[DATE_COLUMN].duplicated().any():
    raise ValueError("Date column contains duplicate timestamps.")

dates = data[DATE_COLUMN].copy()
y = data[TARGET_COLUMN].astype(float).copy()
X = data.drop(columns=[DATE_COLUMN, TARGET_COLUMN]).copy()

non_numeric_columns = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_columns:
    logger.warning("Converting %d non-numeric predictor columns to numeric with coercion.", len(non_numeric_columns))
    for column in non_numeric_columns:
        X[column] = pd.to_numeric(X[column], errors="coerce")

infinite_feature_count = int(np.isinf(X.to_numpy(dtype=float)).sum())
if infinite_feature_count:
    logger.warning("Replacing %d infinite feature values with NaN for median imputation.", infinite_feature_count)
    X = X.replace([np.inf, -np.inf], np.nan)

logger.info("Rows: %d | Predictors: %d | Target: %s", len(data), X.shape[1], TARGET_COLUMN)
logger.info("Date range: %s to %s", dates.min().date(), dates.max().date())
display(data[[DATE_COLUMN, TARGET_COLUMN]].head())
display(X.describe().T[["count", "mean", "std", "min", "max"]].head(10))

## 4. Chronological Train/Test Split

This is a forecasting problem, so the test set is the most recent 20% of observations. No random shuffling is used anywhere in the split or cross-validation workflow.


In [ ]:
split_index = int(len(data) * (1 - TEST_SIZE))
if split_index <= CV_SPLITS + 5 or split_index >= len(data):
    raise ValueError("Dataset is too small for the configured chronological split and TimeSeriesSplit.")

X_train = X.iloc[:split_index].copy()
y_train = y.iloc[:split_index].copy()
dates_train = dates.iloc[:split_index].copy()

X_test = X.iloc[split_index:].copy()
y_test = y.iloc[split_index:].copy()
dates_test = dates.iloc[split_index:].copy()

logger.info("Training rows: %d (%s to %s)", len(X_train), dates_train.min().date(), dates_train.max().date())
logger.info("Test rows: %d (%s to %s)", len(X_test), dates_test.min().date(), dates_test.max().date())

pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(X_train), len(X_test)],
        "start_date": [dates_train.min(), dates_test.min()],
        "end_date": [dates_train.max(), dates_test.max()],
    }
)


## 5. Stage B Model-Based Feature Selection

A Random Forest Regressor is trained on the chronological training set using all predictors. Its feature importances are used only to rank features for benchmarking top-N subsets. This is model-based feature selection, not correlation filtering.


In [ ]:
import time

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

required_variables = {
    "RANDOM_SEED": "the imports/configuration cell",
    "X_train": "the chronological train/test split cell",
    "y_train": "the chronological train/test split cell",
    "FEATURE_IMPORTANCE_PATH": "the imports/configuration cell",
}
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    missing_steps = sorted({required_variables[name] for name in missing_variables})
    raise RuntimeError(
        "Cell 5 requires earlier cells to run first. Missing: "
        + ", ".join(missing_variables)
        + ". Run the notebook from the top, or run: "
        + " -> ".join(missing_steps)
        + "."
    )


def make_random_forest(n_estimators: int = 400) -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        min_samples_leaf=2,
        max_features="sqrt",
    )


def make_lgbm() -> LGBMRegressor:
    return LGBMRegressor(
        n_estimators=500,
        learning_rate=0.03,
        num_leaves=31,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbose=-1,
    )


def make_pipeline(model):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("model", model),
        ]
    )

logger.info("Training initial Random Forest on all %d predictors for feature ranking", X_train.shape[1])
importance_pipeline = make_pipeline(make_random_forest(n_estimators=500))
importance_start = time.perf_counter()
importance_pipeline.fit(X_train, y_train)
importance_time = time.perf_counter() - importance_start
logger.info("Feature ranking model trained in %.2f seconds", importance_time)

feature_importance = pd.DataFrame(
    {
        "feature": X_train.columns,
        "importance": importance_pipeline.named_steps["model"].feature_importances_,
    }
).sort_values("importance", ascending=False, kind="mergesort").reset_index(drop=True)
feature_importance.insert(0, "rank", np.arange(1, len(feature_importance) + 1))
feature_importance.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

logger.info("Saved feature importance ranking to %s", FEATURE_IMPORTANCE_PATH)
display(feature_importance.head(25))

## 6. TimeSeriesSplit Cross-Validation Experiments

Each top-N feature subset is evaluated with two baseline models: Random Forest Regressor and LightGBM Regressor. Metrics are averaged across expanding-window `TimeSeriesSplit` folds.


In [ ]:
def compute_metrics(y_true, y_pred) -> dict[str, float]:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return {"mae": mae, "rmse": rmse, "r2": r2}

def evaluate_model_cv(model_name: str, model_factory: Callable[[], object], features: list[str], subset_label: str) -> dict[str, float | str | int]:
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
    fold_rows: list[dict[str, float]] = []
    X_subset = X_train[features]

    for fold, (train_index, valid_index) in enumerate(tscv.split(X_subset), start=1):
        X_fold_train = X_subset.iloc[train_index]
        y_fold_train = y_train.iloc[train_index]
        X_fold_valid = X_subset.iloc[valid_index]
        y_fold_valid = y_train.iloc[valid_index]

        pipeline = make_pipeline(model_factory())
        start_time = time.perf_counter()
        pipeline.fit(X_fold_train, y_fold_train)
        train_time = time.perf_counter() - start_time
        predictions = pipeline.predict(X_fold_valid)
        metrics = compute_metrics(y_fold_valid, predictions)
        fold_rows.append({**metrics, "training_time_seconds": train_time})

    fold_metrics = pd.DataFrame(fold_rows)
    return {
        "model": model_name,
        "feature_subset": subset_label,
        "n_features": len(features),
        "mae": float(fold_metrics["mae"].mean()),
        "mae_std": float(fold_metrics["mae"].std(ddof=0)),
        "rmse": float(fold_metrics["rmse"].mean()),
        "rmse_std": float(fold_metrics["rmse"].std(ddof=0)),
        "r2": float(fold_metrics["r2"].mean()),
        "r2_std": float(fold_metrics["r2"].std(ddof=0)),
        "training_time_seconds": float(fold_metrics["training_time_seconds"].sum()),
    }

model_factories: dict[str, Callable[[], object]] = {
    "RandomForestRegressor": lambda: make_random_forest(n_estimators=300),
    "LightGBMRegressor": make_lgbm,
}

experiment_results: list[dict[str, float | str | int]] = []
ranked_features = feature_importance["feature"].tolist()
for top_n in TOP_N_OPTIONS:
    selected = ranked_features[: min(top_n, len(ranked_features))]
    subset_label = f"top_{top_n}"
    logger.info("Evaluating subset %s with %d features", subset_label, len(selected))
    for model_name, factory in model_factories.items():
        logger.info("  Model: %s", model_name)
        result = evaluate_model_cv(model_name, factory, selected, subset_label)
        experiment_results.append(result)

model_comparison = pd.DataFrame(experiment_results).sort_values(
    ["rmse", "mae"], ascending=[True, True], kind="mergesort"
).reset_index(drop=True)
model_comparison.insert(0, "rank", np.arange(1, len(model_comparison) + 1))
model_comparison.to_csv(MODEL_COMPARISON_PATH, index=False)

logger.info("Saved model comparison table to %s", MODEL_COMPARISON_PATH)
display(model_comparison)


## 7. Select and Retrain the Winning Baseline

The best combination is selected by lowest cross-validated RMSE, with MAE as the tie-breaker. The winning model is then retrained on the full chronological training set using only its selected feature subset.


In [ ]:
best_row = model_comparison.iloc[0].copy()
best_model_name = str(best_row["model"])
best_n_features = int(best_row["n_features"])
best_subset_label = str(best_row["feature_subset"])
selected_features = ranked_features[:best_n_features]

selected_features_df = pd.DataFrame(
    {
        "selected_rank": np.arange(1, len(selected_features) + 1),
        "feature": selected_features,
    }
).merge(feature_importance[["feature", "rank", "importance"]], on="feature", how="left")
selected_features_df.to_csv(SELECTED_FEATURES_PATH, index=False)
logger.info("Saved selected feature list to %s", SELECTED_FEATURES_PATH)

best_factory = model_factories[best_model_name]
final_model = make_pipeline(best_factory())
logger.info("Retraining winning model: %s with %d features", best_model_name, best_n_features)
final_train_start = time.perf_counter()
final_model.fit(X_train[selected_features], y_train)
final_training_time = time.perf_counter() - final_train_start
logger.info("Final model trained in %.2f seconds", final_training_time)


## 8. Held-Out Test Evaluation

The final test set is the most recent chronological block and is not used for feature ranking, cross-validation, or model selection.


In [ ]:
test_predictions = final_model.predict(X_test[selected_features])
residuals = y_test.to_numpy() - test_predictions
final_metrics = compute_metrics(y_test, test_predictions)
final_metrics["training_time_seconds"] = final_training_time

final_metrics_df = pd.DataFrame(
    [
        {
            "model": best_model_name,
            "feature_subset": best_subset_label,
            "n_features": best_n_features,
            "test_mae": final_metrics["mae"],
            "test_rmse": final_metrics["rmse"],
            "test_r2": final_metrics["r2"],
            "training_time_seconds": final_metrics["training_time_seconds"],
        }
    ]
)
final_metrics_df.to_csv(FINAL_METRICS_PATH, index=False)
logger.info("Saved final test metrics to %s", FINAL_METRICS_PATH)
display(final_metrics_df)

prediction_frame = pd.DataFrame(
    {
        DATE_COLUMN: dates_test.to_numpy(),
        "actual": y_test.to_numpy(),
        "predicted": test_predictions,
        "residual": residuals,
    }
)
display(prediction_frame.head())


## 9. Diagnostic Plots

The notebook saves actual-vs-predicted, residual, feature-importance, and learning-curve figures into `output/analysis/figures/ml`.


In [ ]:
def save_current_figure(filename: str) -> Path:
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    logger.info("Saved figure: %s", path)
    plt.show()
    return path

plt.figure(figsize=(12, 5))
plt.plot(prediction_frame[DATE_COLUMN], prediction_frame["actual"], label="Actual", linewidth=2)
plt.plot(prediction_frame[DATE_COLUMN], prediction_frame["predicted"], label="Predicted", linewidth=2)
plt.title("Food Price Inflation: Actual vs Predicted")
plt.xlabel("Date")
plt.ylabel("Food price inflation")
plt.legend()
plt.grid(alpha=0.25)
actual_predicted_path = save_current_figure("stageB_actual_vs_predicted.png")

plt.figure(figsize=(12, 4))
plt.axhline(0, color="black", linewidth=1)
plt.plot(prediction_frame[DATE_COLUMN], prediction_frame["residual"], marker="o", linewidth=1)
plt.title("Residuals Over Time")
plt.xlabel("Date")
plt.ylabel("Actual - Predicted")
plt.grid(alpha=0.25)
residual_time_path = save_current_figure("stageB_residuals_over_time.png")

plt.figure(figsize=(8, 4))
plt.hist(prediction_frame["residual"], bins=20, edgecolor="black", alpha=0.8)
plt.title("Residual Distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")
residual_hist_path = save_current_figure("stageB_residual_distribution.png")

final_estimator = final_model.named_steps["model"]
if hasattr(final_estimator, "feature_importances_"):
    final_importance = pd.DataFrame(
        {
            "feature": selected_features,
            "importance": final_estimator.feature_importances_,
        }
    ).sort_values("importance", ascending=False, kind="mergesort")
    top_plot = final_importance.head(25).sort_values("importance", ascending=True)
    plt.figure(figsize=(10, 8))
    plt.barh(top_plot["feature"], top_plot["importance"])
    plt.title(f"Top 25 Final Model Importances ({best_model_name})")
    plt.xlabel("Importance")
    feature_importance_plot_path = save_current_figure("stageB_final_model_feature_importance.png")
else:
    logger.warning("Winning model does not expose feature_importances_; skipping importance plot.")
    feature_importance_plot_path = None


## 10. Learning Curve

This time-aware learning curve trains on growing prefixes of the training period and evaluates on the final validation block within the training period. It does not shuffle observations.


In [ ]:
validation_start = int(len(X_train) * 0.80)
X_learning_train = X_train.iloc[:validation_start][selected_features]
y_learning_train = y_train.iloc[:validation_start]
X_learning_valid = X_train.iloc[validation_start:][selected_features]
y_learning_valid = y_train.iloc[validation_start:]

learning_rows = []
train_sizes = np.linspace(0.30, 1.00, 6)
for fraction in train_sizes:
    n_rows = max(12, int(len(X_learning_train) * fraction))
    model = make_pipeline(best_factory())
    start_time = time.perf_counter()
    model.fit(X_learning_train.iloc[:n_rows], y_learning_train.iloc[:n_rows])
    train_time = time.perf_counter() - start_time
    train_pred = model.predict(X_learning_train.iloc[:n_rows])
    valid_pred = model.predict(X_learning_valid)
    learning_rows.append(
        {
            "training_rows": n_rows,
            "train_rmse": math.sqrt(mean_squared_error(y_learning_train.iloc[:n_rows], train_pred)),
            "validation_rmse": math.sqrt(mean_squared_error(y_learning_valid, valid_pred)),
            "training_time_seconds": train_time,
        }
    )

learning_curve = pd.DataFrame(learning_rows)
display(learning_curve)

plt.figure(figsize=(8, 5))
plt.plot(learning_curve["training_rows"], learning_curve["train_rmse"], marker="o", label="Train RMSE")
plt.plot(learning_curve["training_rows"], learning_curve["validation_rmse"], marker="o", label="Validation RMSE")
plt.title("Time-Aware Learning Curve")
plt.xlabel("Training rows")
plt.ylabel("RMSE")
plt.legend()
plt.grid(alpha=0.25)
learning_curve_path = save_current_figure("stageB_learning_curve.png")


## 11. Save Trained Model Artifact

The saved artifact includes the fitted preprocessing/model pipeline, selected feature names, target/date metadata, and final test metrics.


In [ ]:
model_artifact = {
    "pipeline": final_model,
    "model_name": best_model_name,
    "selected_features": selected_features,
    "target_column": TARGET_COLUMN,
    "date_column": DATE_COLUMN,
    "feature_subset": best_subset_label,
    "n_features": best_n_features,
    "cv_selection_row": best_row.to_dict(),
    "final_test_metrics": final_metrics_df.iloc[0].to_dict(),
    "random_seed": RANDOM_SEED,
}
joblib.dump(model_artifact, MODEL_PATH)
logger.info("Saved trained model artifact to %s", MODEL_PATH)
MODEL_PATH


## 12. Final Summary

This cell prints the winning baseline, selected feature count, final held-out test metrics, and artifact locations.


In [ ]:
summary = {
    "best_model": best_model_name,
    "optimal_feature_subset": best_subset_label,
    "selected_feature_count": best_n_features,
    "cv_rmse": float(best_row["rmse"]),
    "cv_mae": float(best_row["mae"]),
    "test_rmse": float(final_metrics["rmse"]),
    "test_mae": float(final_metrics["mae"]),
    "test_r2": float(final_metrics["r2"]),
    "model_path": str(MODEL_PATH),
    "selected_features_path": str(SELECTED_FEATURES_PATH),
    "feature_importance_path": str(FEATURE_IMPORTANCE_PATH),
    "model_comparison_path": str(MODEL_COMPARISON_PATH),
    "figures_dir": str(FIGURE_DIR),
}
summary_df = pd.DataFrame([summary])
display(summary_df.T.rename(columns={0: "value"}))

print("Stage B complete.")
print(f"Best baseline model: {best_model_name}")
print(f"Optimal number of selected features: {best_n_features}")
print(f"Final test RMSE: {final_metrics['rmse']:.4f}")
print(f"Final test MAE: {final_metrics['mae']:.4f}")
print(f"Final test R2: {final_metrics['r2']:.4f}")


In [ ]:
from pathlib import Path

REPORT_DIR = PROJECT_ROOT / "output" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

best_cv_rmse = float(best_row["rmse"])
best_cv_mae = float(best_row["mae"])
test_rmse = float(final_metrics["rmse"])
test_mae = float(final_metrics["mae"])
test_r2 = float(final_metrics["r2"])
train_rows = len(X_train)
test_rows = len(X_test)

if test_r2 >= 0.75:
    r2_interpretation = "The model explains most of the variation in the held-out test period."
elif test_r2 >= 0.50:
    r2_interpretation = "The model explains a meaningful share of the variation, but substantial error remains."
elif test_r2 >= 0:
    r2_interpretation = "The model explains only a limited share of the variation and should be treated as a baseline."
else:
    r2_interpretation = "The model performs worse than a simple constant-average baseline on the held-out period."

if test_rmse <= best_cv_rmse * 1.10:
    generalization_note = "The held-out error is close to cross-validation error, which suggests reasonable generalization."
else:
    generalization_note = "The held-out error is noticeably higher than cross-validation error, suggesting possible overfitting or a harder test period."

top_features = feature_importance.head(10)[["rank", "feature", "importance"]]
top_features_text = "\n".join(
    f"{int(row.rank)}. {row.feature}: {float(row.importance):.6f}"
    for row in top_features.itertuples(index=False)
)

comparison_columns = ["rank", "model", "feature_subset", "n_features", "rmse", "mae", "r2"]
comparison_text = model_comparison[comparison_columns].head(10).to_string(index=False, float_format=lambda value: f"{value:.4f}")

report = f"""# MmarakaAI Stage B Classical ML Results

## What this experiment did

The notebook used the cleaned feature matrix, sorted the observations by date, and kept the most recent 20% as a held-out test period. The earlier 80% was used for model training, feature ranking, and time-aware cross-validation.

- Dataset rows: {len(data)}
- Training rows: {train_rows}
- Held-out test rows: {test_rows}
- Training period: {dates_train.min().date()} to {dates_train.max().date()}
- Test period: {dates_test.min().date()} to {dates_test.max().date()}
- Target: `{TARGET_COLUMN}`
- Candidate models: Random Forest and LightGBM
- Feature subsets tested: {", ".join(str(value) for value in TOP_N_OPTIONS)} top-ranked features

## Main result

- Best model: **{best_model_name}**
- Selected feature subset: **{best_subset_label}**
- Selected feature count: **{best_n_features}**
- Cross-validation RMSE: **{best_cv_rmse:.4f}**
- Cross-validation MAE: **{best_cv_mae:.4f}**
- Held-out test RMSE: **{test_rmse:.4f}**
- Held-out test MAE: **{test_mae:.4f}**
- Held-out test R-squared: **{test_r2:.4f}**

## Plain-language interpretation

### RMSE
RMSE is the typical prediction error, with larger errors receiving extra penalty. Lower is better. The held-out RMSE of **{test_rmse:.4f}** means the model's prediction errors are approximately this size on the scale of the target, with larger misses affecting the score more strongly.

### MAE
MAE is the average absolute prediction error. Lower is better, and it is easier to interpret than RMSE because it is not as strongly affected by unusually large errors. The held-out MAE is **{test_mae:.4f}**.

### R-squared
R-squared compares the model with a baseline that always predicts the training-period average. A value of 1 is perfect, 0 means no improvement over that constant baseline, and a negative value means the model is worse than that baseline. The test R-squared is **{test_r2:.4f}**. {r2_interpretation}

## Generalization check

{generalization_note}

Cross-validation estimates performance on several earlier validation windows. The held-out test result is the more important final check because it uses the latest observations and was not used to choose the model.

## Top features by Random Forest importance

Feature importance measures how much the fitted Random Forest used each feature to reduce prediction error. It indicates usefulness within this model; it does not prove that a feature causes inflation, and it should not be read as a percentage of inflation caused by that feature.

{top_features_text}

## Model comparison

The table below shows the ten highest-ranked model/subset combinations. Lower RMSE and MAE are better; higher R-squared is better.

```text
{comparison_text}
```

## Files produced

- Model artifact: `{MODEL_PATH}`
- Feature ranking: `{FEATURE_IMPORTANCE_PATH}`
- Model comparison: `{MODEL_COMPARISON_PATH}`
- Selected features: `{SELECTED_FEATURES_PATH}`
- Figures directory: `{FIGURE_DIR}`

## Important limitation

This is a baseline forecasting experiment, not proof that the model will perform equally well in the future. The dataset contains engineered lag, rolling, and summary features, so performance should be checked for leakage and stability before using the model operationally.
"""

REPORT_MD_PATH = REPORT_DIR / "classic_ml_results.md"
REPORT_TXT_PATH = REPORT_DIR / "classic_ml_results.txt"
REPORT_MD_PATH.write_text(report, encoding="utf-8")
REPORT_TXT_PATH.write_text(report, encoding="utf-8")

print(f"Readable Markdown report: {REPORT_MD_PATH}")
print(f"Plain-text report: {REPORT_TXT_PATH}")


## 13. Stage C Hyperparameter Optimization Overview

Stage C keeps the Stage B preprocessing, engineered features, chronological train/test split, and original Random Forest feature-ranking baseline intact. It adds a second model-based feature-ranking method using LightGBM importance, compares feature subset sizes across both ranking methods, then tunes LightGBM and Random Forest with `TimeSeriesSplit` only.

The fixed Stage B reference model is **LightGBMRegressor with the top 100 Random-Forest-ranked features**. Stage C model selection is based on cross-validation RMSE first and MAE second; the held-out test set remains a final evaluation only.


In [ ]:
import json
from pathlib import Path

required_stagec_variables = {
    "PROJECT_ROOT": "configuration/path cell",
    "X_train": "chronological train/test split cell",
    "y_train": "chronological train/test split cell",
    "X_test": "chronological train/test split cell",
    "y_test": "chronological train/test split cell",
    "dates_test": "chronological train/test split cell",
    "feature_importance": "Stage B feature-selection cell",
    "model_factories": "Stage B cross-validation cell",
    "make_pipeline": "Stage B feature-selection cell",
    "make_lgbm": "Stage B feature-selection cell",
    "make_random_forest": "Stage B feature-selection cell",
    "compute_metrics": "Stage B cross-validation cell",
    "evaluate_model_cv": "Stage B cross-validation cell",
}
missing_stagec_variables = [name for name in required_stagec_variables if name not in globals()]
if missing_stagec_variables:
    raise RuntimeError(
        "Stage C requires the Stage B notebook cells to run first. Missing: "
        + ", ".join(missing_stagec_variables)
    )

STAGEB_REFERENCE_MODEL_NAME = "LightGBMRegressor"
STAGEB_REFERENCE_RANKING_METHOD = "RandomForestImportance"
STAGEB_REFERENCE_N_FEATURES = 100

STAGEC_RANDOM_SEARCH_ITERATIONS = 24
STAGEC_TOP_N_OPTIONS = list(TOP_N_OPTIONS)
STAGEC_SEARCH_SCORING = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

REPORT_DIR = OUTPUT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

STAGEC_MODEL_PATH = MODELS_DIR / "stageC_best_model.joblib"
STAGEC_BEST_HYPERPARAMETERS_PATH = OUTPUT_DIR / "best_hyperparameters.json"
STAGEC_FEATURE_RANKINGS_PATH = OUTPUT_DIR / "stageC_feature_rankings.csv"
STAGEC_SELECTED_FEATURES_PATH = OUTPUT_DIR / "stageC_selected_features.csv"
STAGEC_FEATURE_SELECTION_COMPARISON_PATH = OUTPUT_DIR / "stageC_feature_selection_comparison.csv"
STAGEC_SEARCH_RESULTS_PATH = OUTPUT_DIR / "stageC_hyperparameter_search_results.csv"
STAGEC_HYPERPARAMETER_EFFECTS_PATH = OUTPUT_DIR / "stageC_hyperparameter_effects.csv"
STAGEC_MODEL_COMPARISON_PATH = OUTPUT_DIR / "stageC_model_comparison.csv"
STAGEC_TEST_PREDICTIONS_PATH = OUTPUT_DIR / "stageC_test_predictions.csv"
STAGEC_LEARNING_CURVE_PATH = OUTPUT_DIR / "stageC_learning_curve.csv"
STAGEC_BOOTSTRAP_PATH = OUTPUT_DIR / "stageC_bootstrap_improvement.csv"
STAGEC_REPORT_PATH = REPORT_DIR / "stageC_hyperparameter_optimization_report.md"

logger.info("Stage C output directory: %s", OUTPUT_DIR)
logger.info("Stage C figure directory: %s", FIGURE_DIR)


## 14. Stage C Feature-Ranking Comparison

The current Stage B workflow ranks features with Random Forest importance. Stage C adds a LightGBM importance ranking, evaluates both rankings over the same top-N subset sizes, and lets each model family choose the feature set that performs best under `TimeSeriesSplit`.


In [ ]:
logger.info("Training LightGBM feature-ranking model on all training predictors")
lgbm_ranking_pipeline = make_pipeline(make_lgbm())
lgbm_ranking_start = time.perf_counter()
lgbm_ranking_pipeline.fit(X_train, y_train)
lgbm_ranking_time = time.perf_counter() - lgbm_ranking_start
logger.info("LightGBM feature-ranking model trained in %.2f seconds", lgbm_ranking_time)

rf_feature_ranking = feature_importance[["rank", "feature", "importance"]].copy()
rf_feature_ranking.insert(0, "ranking_method", "RandomForestImportance")

lgbm_feature_ranking = pd.DataFrame(
    {
        "feature": X_train.columns,
        "importance": lgbm_ranking_pipeline.named_steps["model"].feature_importances_,
    }
).sort_values("importance", ascending=False, kind="mergesort").reset_index(drop=True)
lgbm_feature_ranking.insert(0, "rank", np.arange(1, len(lgbm_feature_ranking) + 1))
lgbm_feature_ranking.insert(0, "ranking_method", "LightGBMImportance")

stagec_feature_rankings = pd.concat([rf_feature_ranking, lgbm_feature_ranking], ignore_index=True)
stagec_feature_rankings.to_csv(STAGEC_FEATURE_RANKINGS_PATH, index=False)
logger.info("Saved Stage C feature rankings to %s", STAGEC_FEATURE_RANKINGS_PATH)

feature_rankings_by_method = {
    "RandomForestImportance": rf_feature_ranking.sort_values("rank")["feature"].tolist(),
    "LightGBMImportance": lgbm_feature_ranking.sort_values("rank")["feature"].tolist(),
}

stageb_reference_features = feature_rankings_by_method[STAGEB_REFERENCE_RANKING_METHOD][
    : min(STAGEB_REFERENCE_N_FEATURES, len(feature_rankings_by_method[STAGEB_REFERENCE_RANKING_METHOD]))
]

feature_selection_rows = []
for ranking_method, ranked_feature_names in feature_rankings_by_method.items():
    for top_n in STAGEC_TOP_N_OPTIONS:
        selected_stagec_features = ranked_feature_names[: min(top_n, len(ranked_feature_names))]
        subset_label = f"{ranking_method}_top_{len(selected_stagec_features)}"
        logger.info("Stage C feature-selection comparison: %s", subset_label)
        for model_name, model_factory in model_factories.items():
            row = evaluate_model_cv(model_name, model_factory, selected_stagec_features, subset_label)
            row["ranking_method"] = ranking_method
            row["requested_top_n"] = top_n
            feature_selection_rows.append(row)

stagec_feature_selection_comparison = pd.DataFrame(feature_selection_rows).sort_values(
    ["rmse", "mae"], ascending=[True, True], kind="mergesort"
).reset_index(drop=True)
stagec_feature_selection_comparison.insert(0, "stagec_feature_selection_rank", np.arange(1, len(stagec_feature_selection_comparison) + 1))
stagec_feature_selection_comparison.to_csv(STAGEC_FEATURE_SELECTION_COMPARISON_PATH, index=False)
logger.info("Saved Stage C feature-selection comparison to %s", STAGEC_FEATURE_SELECTION_COMPARISON_PATH)

stagec_model_feature_choices = {}
stagec_selected_feature_rows = []
for model_name in model_factories:
    model_rows = stagec_feature_selection_comparison[stagec_feature_selection_comparison["model"] == model_name]
    if model_rows.empty:
        raise RuntimeError(f"No Stage C feature-selection results found for {model_name}")
    best_model_feature_row = model_rows.iloc[0].copy()
    ranking_method = str(best_model_feature_row["ranking_method"])
    n_features = int(best_model_feature_row["n_features"])
    features = feature_rankings_by_method[ranking_method][:n_features]
    stagec_model_feature_choices[model_name] = {
        "ranking_method": ranking_method,
        "n_features": n_features,
        "feature_subset": str(best_model_feature_row["feature_subset"]),
        "features": features,
        "cv_rmse": float(best_model_feature_row["rmse"]),
        "cv_mae": float(best_model_feature_row["mae"]),
        "cv_r2": float(best_model_feature_row["r2"]),
    }
    for feature_rank, feature_name in enumerate(features, start=1):
        stagec_selected_feature_rows.append(
            {
                "model": model_name,
                "ranking_method": ranking_method,
                "feature_subset": best_model_feature_row["feature_subset"],
                "selected_rank": feature_rank,
                "feature": feature_name,
            }
        )

stagec_selected_features_df = pd.DataFrame(stagec_selected_feature_rows)
stagec_selected_features_df.to_csv(STAGEC_SELECTED_FEATURES_PATH, index=False)

logger.info("Best Stage C feature choices by model: %s", stagec_model_feature_choices)
display(stagec_feature_rankings.groupby("ranking_method").head(15))
display(stagec_feature_selection_comparison.head(20))


## 15. Stage C Hyperparameter Search

For each model family, Stage C uses the best feature-ranking/subset combination found above. Each model first receives a broad `RandomizedSearchCV`, then a smaller focused `GridSearchCV` around the best random-search region. Both searches use `TimeSeriesSplit`; no shuffled validation is used.


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV


def make_stagec_estimator(model_name: str):
    if model_name == "LightGBMRegressor":
        return LGBMRegressor(
            objective="regression",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbose=-1,
            subsample_freq=1,
        )
    if model_name == "RandomForestRegressor":
        return RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)
    raise ValueError(f"Unsupported Stage C model: {model_name}")


def make_stagec_pipeline(model_name: str, clean_params: dict | None = None):
    estimator = make_stagec_estimator(model_name)
    if clean_params:
        estimator.set_params(**clean_params)
    return make_pipeline(estimator)


def strip_model_prefix(params: dict) -> dict:
    return {key.replace("model__", ""): value for key, value in params.items()}


def parameter_space_size(space: dict[str, list]) -> int:
    size = 1
    for values in space.values():
        size *= len(values)
    return size


def distance_to_best(value, best_value) -> float:
    if value is None or best_value is None:
        return 0.0 if value is best_value else 1_000_000.0
    try:
        return abs(float(value) - float(best_value))
    except Exception:
        return 0.0 if value == best_value else 1_000_000.0


def nearby_values(candidates: list, best_value, max_count: int = 2) -> list:
    values = list(dict.fromkeys(candidates + [best_value]))
    values = sorted(values, key=lambda value: (distance_to_best(value, best_value), str(value)))
    chosen = values[:max_count]
    if best_value not in chosen:
        chosen = [best_value, *chosen[:-1]]
    return list(dict.fromkeys(chosen))


def scaled_float_values(best_value: float, lower: float, upper: float, multipliers: tuple[float, ...] = (0.75, 1.25)) -> list[float]:
    values = [float(best_value)]
    for multiplier in multipliers:
        values.append(min(upper, max(lower, float(best_value) * multiplier)))
    return sorted({round(value, 5) for value in values})[:2]


lgbm_random_space = {
    "model__num_leaves": [15, 31, 47, 63, 95, 127],
    "model__max_depth": [-1, 3, 4, 5, 6, 8, 10, 12],
    "model__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.10],
    "model__n_estimators": [200, 300, 500, 700, 900, 1200],
    "model__min_child_samples": [5, 10, 20, 30, 50],
    "model__subsample": [0.65, 0.75, 0.85, 0.95, 1.0],
    "model__colsample_bytree": [0.65, 0.75, 0.85, 0.95, 1.0],
    "model__reg_alpha": [0.0, 0.01, 0.05, 0.1, 0.5, 1.0],
    "model__reg_lambda": [0.0, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0],
    "model__min_split_gain": [0.0, 0.001, 0.01, 0.05, 0.1],
}

rf_random_space = {
    "model__n_estimators": [200, 300, 500, 700, 900],
    "model__max_depth": [None, 4, 6, 8, 12, 16, 24],
    "model__min_samples_split": [2, 4, 6, 10, 15],
    "model__min_samples_leaf": [1, 2, 3, 5, 8],
    "model__max_features": ["sqrt", "log2", 0.4, 0.6, 0.8, 1.0],
    "model__bootstrap": [True, False],
}

random_search_spaces = {
    "LightGBMRegressor": lgbm_random_space,
    "RandomForestRegressor": rf_random_space,
}


def build_focused_grid(model_name: str, best_params: dict) -> dict[str, list]:
    if model_name == "LightGBMRegressor":
        return {
            "model__num_leaves": nearby_values(lgbm_random_space["model__num_leaves"], best_params["model__num_leaves"], 2),
            "model__max_depth": nearby_values(lgbm_random_space["model__max_depth"], best_params["model__max_depth"], 2),
            "model__learning_rate": scaled_float_values(best_params["model__learning_rate"], 0.005, 0.15),
            "model__n_estimators": nearby_values(lgbm_random_space["model__n_estimators"], best_params["model__n_estimators"], 2),
            "model__min_child_samples": [best_params["model__min_child_samples"]],
            "model__subsample": [best_params["model__subsample"]],
            "model__colsample_bytree": [best_params["model__colsample_bytree"]],
            "model__reg_alpha": [best_params["model__reg_alpha"]],
            "model__reg_lambda": [best_params["model__reg_lambda"]],
            "model__min_split_gain": [best_params["model__min_split_gain"]],
        }
    if model_name == "RandomForestRegressor":
        return {
            "model__n_estimators": nearby_values(rf_random_space["model__n_estimators"], best_params["model__n_estimators"], 2),
            "model__max_depth": nearby_values(rf_random_space["model__max_depth"], best_params["model__max_depth"], 2),
            "model__min_samples_split": [best_params["model__min_samples_split"]],
            "model__min_samples_leaf": nearby_values(rf_random_space["model__min_samples_leaf"], best_params["model__min_samples_leaf"], 2),
            "model__max_features": nearby_values(rf_random_space["model__max_features"], best_params["model__max_features"], 2),
            "model__bootstrap": [best_params["model__bootstrap"]],
        }
    raise ValueError(f"Unsupported Stage C model: {model_name}")


def cv_results_to_frame(search, model_name: str, search_stage: str, feature_info: dict, elapsed_seconds: float) -> pd.DataFrame:
    frame = pd.DataFrame(search.cv_results_).copy()
    frame["model"] = model_name
    frame["search_stage"] = search_stage
    frame["ranking_method"] = feature_info["ranking_method"]
    frame["feature_subset"] = feature_info["feature_subset"]
    frame["n_features"] = feature_info["n_features"]
    frame["search_elapsed_seconds"] = elapsed_seconds
    frame["cv_rmse"] = -frame["mean_test_rmse"]
    frame["cv_mae"] = -frame["mean_test_mae"]
    frame["cv_r2"] = frame["mean_test_r2"]
    if "mean_train_rmse" in frame.columns:
        frame["train_rmse"] = -frame["mean_train_rmse"]
    return frame


def summarize_search(search, model_name: str, search_stage: str, feature_info: dict, elapsed_seconds: float) -> dict:
    best_index = int(search.best_index_)
    cv_frame = pd.DataFrame(search.cv_results_)
    best_cv = cv_frame.iloc[best_index]
    features = feature_info["features"]
    predictions = search.best_estimator_.predict(X_test[features])
    metrics = compute_metrics(y_test, predictions)
    row = {
        "version": f"StageC_{search_stage}",
        "model": model_name,
        "search_stage": search_stage,
        "ranking_method": feature_info["ranking_method"],
        "feature_subset": feature_info["feature_subset"],
        "n_features": int(feature_info["n_features"]),
        "cv_rmse": float(-best_cv["mean_test_rmse"]),
        "cv_rmse_std": float(best_cv["std_test_rmse"]),
        "cv_mae": float(-best_cv["mean_test_mae"]),
        "cv_mae_std": float(best_cv["std_test_mae"]),
        "cv_r2": float(best_cv["mean_test_r2"]),
        "cv_r2_std": float(best_cv["std_test_r2"]),
        "train_rmse": float(-best_cv["mean_train_rmse"]) if "mean_train_rmse" in best_cv else np.nan,
        "test_rmse": float(metrics["rmse"]),
        "test_mae": float(metrics["mae"]),
        "test_r2": float(metrics["r2"]),
        "training_time_seconds": float(elapsed_seconds),
        "best_params": strip_model_prefix(search.best_params_),
    }
    prediction_frame = pd.DataFrame(
        {
            DATE_COLUMN: dates_test.to_numpy(),
            "actual": y_test.to_numpy(),
            "predicted": predictions,
            "residual": y_test.to_numpy() - predictions,
            "model": model_name,
            "search_stage": search_stage,
        }
    )
    return row, prediction_frame


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(val) for key, val in value.items()}
    if isinstance(value, list):
        return [json_safe(item) for item in value]
    if isinstance(value, tuple):
        return [json_safe(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if pd.isna(value) if not isinstance(value, (list, tuple, dict, str)) else False:
        return None
    return value


# Recompute the fixed Stage B reference explicitly: LightGBM + top 100 RF-ranked features.
baseline_cv_match = model_comparison[
    (model_comparison["model"] == STAGEB_REFERENCE_MODEL_NAME)
    & (model_comparison["feature_subset"] == f"top_{STAGEB_REFERENCE_N_FEATURES}")
]
if baseline_cv_match.empty:
    logger.warning("Stage B reference row was not found in model_comparison; recomputing CV metrics.")
    baseline_cv_row = evaluate_model_cv(
        STAGEB_REFERENCE_MODEL_NAME,
        make_lgbm,
        stageb_reference_features,
        f"top_{len(stageb_reference_features)}",
    )
else:
    baseline_cv_row = baseline_cv_match.iloc[0].to_dict()

stageb_reference_pipeline = make_pipeline(make_lgbm())
baseline_fit_start = time.perf_counter()
stageb_reference_pipeline.fit(X_train[stageb_reference_features], y_train)
stageb_reference_training_time = time.perf_counter() - baseline_fit_start
stageb_reference_predictions = stageb_reference_pipeline.predict(X_test[stageb_reference_features])
stageb_reference_test_metrics = compute_metrics(y_test, stageb_reference_predictions)
stageb_reference_prediction_frame = pd.DataFrame(
    {
        DATE_COLUMN: dates_test.to_numpy(),
        "actual": y_test.to_numpy(),
        "predicted": stageb_reference_predictions,
        "residual": y_test.to_numpy() - stageb_reference_predictions,
        "model": STAGEB_REFERENCE_MODEL_NAME,
        "search_stage": "stageB_reference",
    }
)

stagec_comparison_rows = [
    {
        "version": "StageB_reference",
        "model": STAGEB_REFERENCE_MODEL_NAME,
        "search_stage": "baseline_reference",
        "ranking_method": STAGEB_REFERENCE_RANKING_METHOD,
        "feature_subset": f"top_{len(stageb_reference_features)}",
        "n_features": len(stageb_reference_features),
        "cv_rmse": float(baseline_cv_row["rmse"]),
        "cv_rmse_std": float(baseline_cv_row.get("rmse_std", np.nan)),
        "cv_mae": float(baseline_cv_row["mae"]),
        "cv_mae_std": float(baseline_cv_row.get("mae_std", np.nan)),
        "cv_r2": float(baseline_cv_row["r2"]),
        "cv_r2_std": float(baseline_cv_row.get("r2_std", np.nan)),
        "train_rmse": np.nan,
        "test_rmse": float(stageb_reference_test_metrics["rmse"]),
        "test_mae": float(stageb_reference_test_metrics["mae"]),
        "test_r2": float(stageb_reference_test_metrics["r2"]),
        "training_time_seconds": float(stageb_reference_training_time),
        "best_params": make_lgbm().get_params(),
    }
]

stagec_search_objects = {}
stagec_prediction_frames = []
stagec_cv_result_frames = []

for model_name in ["LightGBMRegressor", "RandomForestRegressor"]:
    feature_info = stagec_model_feature_choices[model_name]
    features = feature_info["features"]
    X_search = X_train[features]
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
    random_space = random_search_spaces[model_name]
    n_iter = min(STAGEC_RANDOM_SEARCH_ITERATIONS, parameter_space_size(random_space))

    logger.info("Stage C randomized search: %s | %d features | %d iterations", model_name, len(features), n_iter)
    random_search = RandomizedSearchCV(
        estimator=make_stagec_pipeline(model_name),
        param_distributions=random_space,
        n_iter=n_iter,
        scoring=STAGEC_SEARCH_SCORING,
        refit="rmse",
        cv=tscv,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbose=1,
        return_train_score=True,
        error_score="raise",
    )
    random_start = time.perf_counter()
    random_search.fit(X_search, y_train)
    random_elapsed = time.perf_counter() - random_start
    stagec_search_objects[(model_name, "randomized_search")] = random_search
    random_summary, random_predictions = summarize_search(random_search, model_name, "randomized_search", feature_info, random_elapsed)
    stagec_comparison_rows.append(random_summary)
    stagec_prediction_frames.append(random_predictions)
    stagec_cv_result_frames.append(cv_results_to_frame(random_search, model_name, "randomized_search", feature_info, random_elapsed))

    focused_grid = build_focused_grid(model_name, random_search.best_params_)
    logger.info("Stage C focused grid search: %s | grid=%s", model_name, focused_grid)
    grid_search = GridSearchCV(
        estimator=make_stagec_pipeline(model_name),
        param_grid=focused_grid,
        scoring=STAGEC_SEARCH_SCORING,
        refit="rmse",
        cv=TimeSeriesSplit(n_splits=CV_SPLITS),
        n_jobs=-1,
        verbose=1,
        return_train_score=True,
        error_score="raise",
    )
    grid_start = time.perf_counter()
    grid_search.fit(X_search, y_train)
    grid_elapsed = time.perf_counter() - grid_start
    stagec_search_objects[(model_name, "focused_grid_search")] = grid_search
    grid_summary, grid_predictions = summarize_search(grid_search, model_name, "focused_grid_search", feature_info, grid_elapsed)
    stagec_comparison_rows.append(grid_summary)
    stagec_prediction_frames.append(grid_predictions)
    stagec_cv_result_frames.append(cv_results_to_frame(grid_search, model_name, "focused_grid_search", feature_info, grid_elapsed))

stagec_hyperparameter_search_results = pd.concat(stagec_cv_result_frames, ignore_index=True)
stagec_hyperparameter_search_results.to_csv(STAGEC_SEARCH_RESULTS_PATH, index=False)
logger.info("Saved Stage C search results to %s", STAGEC_SEARCH_RESULTS_PATH)

stagec_model_comparison = pd.DataFrame(stagec_comparison_rows).sort_values(
    ["cv_rmse", "cv_mae"], ascending=[True, True], kind="mergesort"
).reset_index(drop=True)
stagec_model_comparison.insert(0, "stagec_rank", np.arange(1, len(stagec_model_comparison) + 1))
stagec_model_comparison.to_csv(STAGEC_MODEL_COMPARISON_PATH, index=False)

stagec_tuned_candidates = stagec_model_comparison[stagec_model_comparison["version"] != "StageB_reference"].copy()
stagec_best_row = stagec_tuned_candidates.iloc[0].copy()
stagec_best_model_name = str(stagec_best_row["model"])
stagec_best_search_stage = str(stagec_best_row["search_stage"])
stagec_best_feature_info = stagec_model_feature_choices[stagec_best_model_name]
stagec_best_features = stagec_best_feature_info["features"]
stagec_best_model = stagec_search_objects[(stagec_best_model_name, stagec_best_search_stage)].best_estimator_
stagec_best_clean_params = strip_model_prefix(stagec_search_objects[(stagec_best_model_name, stagec_best_search_stage)].best_params_)

stagec_prediction_frame = pd.concat([stageb_reference_prediction_frame, *stagec_prediction_frames], ignore_index=True)
stagec_prediction_frame.to_csv(STAGEC_TEST_PREDICTIONS_PATH, index=False)

best_hyperparameters_payload = {
    "selection_rule": "Lowest cross-validated RMSE, then lowest cross-validated MAE. Held-out test metrics are reported but not used for model selection.",
    "stageB_reference": {
        "model": STAGEB_REFERENCE_MODEL_NAME,
        "ranking_method": STAGEB_REFERENCE_RANKING_METHOD,
        "n_features": len(stageb_reference_features),
        "params": make_lgbm().get_params(),
        "cv_rmse": float(baseline_cv_row["rmse"]),
        "test_rmse": float(stageb_reference_test_metrics["rmse"]),
    },
    "stageC_best": {
        "model": stagec_best_model_name,
        "search_stage": stagec_best_search_stage,
        "ranking_method": stagec_best_feature_info["ranking_method"],
        "feature_subset": stagec_best_feature_info["feature_subset"],
        "n_features": len(stagec_best_features),
        "params": stagec_best_clean_params,
        "cv_rmse": float(stagec_best_row["cv_rmse"]),
        "cv_mae": float(stagec_best_row["cv_mae"]),
        "test_rmse": float(stagec_best_row["test_rmse"]),
        "test_mae": float(stagec_best_row["test_mae"]),
        "test_r2": float(stagec_best_row["test_r2"]),
    },
    "model_feature_choices": stagec_model_feature_choices,
    "all_comparison_rows": stagec_model_comparison.drop(columns=["best_params"]).to_dict(orient="records"),
}
STAGEC_BEST_HYPERPARAMETERS_PATH.write_text(json.dumps(json_safe(best_hyperparameters_payload), indent=2), encoding="utf-8")

stagec_model_artifact = {
    "pipeline": stagec_best_model,
    "model_name": stagec_best_model_name,
    "search_stage": stagec_best_search_stage,
    "selected_features": stagec_best_features,
    "target_column": TARGET_COLUMN,
    "date_column": DATE_COLUMN,
    "ranking_method": stagec_best_feature_info["ranking_method"],
    "feature_subset": stagec_best_feature_info["feature_subset"],
    "best_params": stagec_best_clean_params,
    "comparison_row": stagec_best_row.to_dict(),
    "stageB_reference_row": stagec_model_comparison[stagec_model_comparison["version"] == "StageB_reference"].iloc[0].to_dict(),
    "random_seed": RANDOM_SEED,
}
joblib.dump(stagec_model_artifact, STAGEC_MODEL_PATH)
logger.info("Saved Stage C best model to %s", STAGEC_MODEL_PATH)

# Hyperparameter effect summary from the broad randomized searches.
effect_rows = []
for model_name in ["LightGBMRegressor", "RandomForestRegressor"]:
    model_random_results = stagec_hyperparameter_search_results[
        (stagec_hyperparameter_search_results["model"] == model_name)
        & (stagec_hyperparameter_search_results["search_stage"] == "randomized_search")
    ].copy()
    for param_column in sorted(column for column in model_random_results.columns if column.startswith("param_model__")):
        grouped = model_random_results.groupby(param_column, dropna=False)["cv_rmse"].mean()
        if len(grouped) <= 1:
            continue
        effect_rows.append(
            {
                "model": model_name,
                "hyperparameter": param_column.replace("param_model__", ""),
                "rmse_effect_range": float(grouped.max() - grouped.min()),
                "best_value_by_mean_rmse": json_safe(grouped.idxmin()),
                "best_mean_cv_rmse": float(grouped.min()),
                "worst_mean_cv_rmse": float(grouped.max()),
                "values_tested": len(grouped),
            }
        )
stagec_hyperparameter_effects = pd.DataFrame(effect_rows).sort_values(
    ["rmse_effect_range"], ascending=False, kind="mergesort"
).reset_index(drop=True)
stagec_hyperparameter_effects.to_csv(STAGEC_HYPERPARAMETER_EFFECTS_PATH, index=False)

# Bootstrap paired test-set improvements. Positive RMSE/MAE deltas mean Stage C improved over Stage B.
def bootstrap_metric_deltas(y_true, baseline_pred, tuned_pred, n_bootstrap: int = 2000) -> pd.DataFrame:
    rng = np.random.default_rng(RANDOM_SEED)
    y_true = np.asarray(y_true, dtype=float)
    baseline_pred = np.asarray(baseline_pred, dtype=float)
    tuned_pred = np.asarray(tuned_pred, dtype=float)
    n = len(y_true)
    rows = []
    for metric_name in ["rmse", "mae", "r2"]:
        deltas = []
        for _ in range(n_bootstrap):
            sample_index = rng.integers(0, n, size=n)
            baseline_metrics = compute_metrics(y_true[sample_index], baseline_pred[sample_index])
            tuned_metrics = compute_metrics(y_true[sample_index], tuned_pred[sample_index])
            if metric_name in ["rmse", "mae"]:
                deltas.append(baseline_metrics[metric_name] - tuned_metrics[metric_name])
            else:
                deltas.append(tuned_metrics[metric_name] - baseline_metrics[metric_name])
        deltas = np.asarray(deltas)
        rows.append(
            {
                "metric": metric_name,
                "mean_delta_improvement": float(deltas.mean()),
                "ci95_lower": float(np.quantile(deltas, 0.025)),
                "ci95_upper": float(np.quantile(deltas, 0.975)),
                "probability_stageC_better": float((deltas > 0).mean()),
            }
        )
    return pd.DataFrame(rows)

stagec_best_predictions = stagec_best_model.predict(X_test[stagec_best_features])
stagec_best_residuals = y_test.to_numpy() - stagec_best_predictions
stageb_reference_residuals = y_test.to_numpy() - stageb_reference_predictions
stagec_bootstrap_improvement = bootstrap_metric_deltas(y_test.to_numpy(), stageb_reference_predictions, stagec_best_predictions)
stagec_bootstrap_improvement.to_csv(STAGEC_BOOTSTRAP_PATH, index=False)

logger.info("Saved Stage C model comparison to %s", STAGEC_MODEL_COMPARISON_PATH)
logger.info("Saved Stage C best hyperparameters to %s", STAGEC_BEST_HYPERPARAMETERS_PATH)
display(stagec_model_comparison)
display(stagec_hyperparameter_effects.head(15))
display(stagec_bootstrap_improvement)


## 16. Stage C Diagnostics and Improvement Figures

These plots compare the fixed Stage B reference against the best tuned Stage C model on the held-out chronological test period. The learning curve remains time-aware and uses growing prefixes of the training period.


In [ ]:
baseline_row = stagec_model_comparison[stagec_model_comparison["version"] == "StageB_reference"].iloc[0]
stagec_best_metrics_row = stagec_model_comparison.iloc[0]

stagec_best_prediction_frame = pd.DataFrame(
    {
        DATE_COLUMN: dates_test.to_numpy(),
        "actual": y_test.to_numpy(),
        "stageB_reference_predicted": stageb_reference_predictions,
        "stageC_tuned_predicted": stagec_best_predictions,
        "stageB_reference_residual": stageb_reference_residuals,
        "stageC_tuned_residual": stagec_best_residuals,
    }
)

plt.figure(figsize=(12, 5))
plt.plot(stagec_best_prediction_frame[DATE_COLUMN], stagec_best_prediction_frame["actual"], label="Actual", linewidth=2)
plt.plot(stagec_best_prediction_frame[DATE_COLUMN], stagec_best_prediction_frame["stageB_reference_predicted"], label="Stage B reference", linewidth=2)
plt.plot(stagec_best_prediction_frame[DATE_COLUMN], stagec_best_prediction_frame["stageC_tuned_predicted"], label="Stage C tuned", linewidth=2)
plt.title("Stage B Reference vs Stage C Tuned Predictions")
plt.xlabel("Date")
plt.ylabel("Food price inflation")
plt.legend()
plt.grid(alpha=0.25)
stagec_prediction_plot_path = save_current_figure("stageC_baseline_vs_tuned_predictions.png")

plt.figure(figsize=(12, 5))
plt.axhline(0, color="black", linewidth=1)
plt.plot(stagec_best_prediction_frame[DATE_COLUMN], stagec_best_prediction_frame["stageB_reference_residual"], label="Stage B residual", linewidth=2)
plt.plot(stagec_best_prediction_frame[DATE_COLUMN], stagec_best_prediction_frame["stageC_tuned_residual"], label="Stage C residual", linewidth=2)
plt.title("Residual Comparison Over Time")
plt.xlabel("Date")
plt.ylabel("Actual - predicted")
plt.legend()
plt.grid(alpha=0.25)
stagec_residual_plot_path = save_current_figure("stageC_residual_comparison.png")

improvement_rows = [
    {"metric": "RMSE", "delta": float(baseline_row["test_rmse"] - stagec_best_metrics_row["test_rmse"]), "direction": "positive is better"},
    {"metric": "MAE", "delta": float(baseline_row["test_mae"] - stagec_best_metrics_row["test_mae"]), "direction": "positive is better"},
    {"metric": "R2", "delta": float(stagec_best_metrics_row["test_r2"] - baseline_row["test_r2"]), "direction": "positive is better"},
]
stagec_improvement_summary = pd.DataFrame(improvement_rows)
plt.figure(figsize=(8, 5))
colors = ["#2f7d32" if value >= 0 else "#b3261e" for value in stagec_improvement_summary["delta"]]
plt.bar(stagec_improvement_summary["metric"], stagec_improvement_summary["delta"], color=colors)
plt.axhline(0, color="black", linewidth=1)
plt.title("Stage C Improvement Over Stage B Reference")
plt.ylabel("Metric delta")
plt.grid(axis="y", alpha=0.25)
stagec_improvement_plot_path = save_current_figure("stageC_improvement_summary.png")

best_estimator = stagec_best_model.named_steps["model"]
if hasattr(best_estimator, "feature_importances_"):
    stagec_best_importance = pd.DataFrame(
        {
            "feature": stagec_best_features,
            "importance": best_estimator.feature_importances_,
        }
    ).sort_values("importance", ascending=False, kind="mergesort").reset_index(drop=True)
    stagec_best_importance.insert(0, "rank", np.arange(1, len(stagec_best_importance) + 1))
    STAGEC_BEST_FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "stageC_best_model_feature_importance.csv"
    stagec_best_importance.to_csv(STAGEC_BEST_FEATURE_IMPORTANCE_PATH, index=False)
    top_stagec_importance = stagec_best_importance.head(25).sort_values("importance", ascending=True)
    plt.figure(figsize=(10, 8))
    plt.barh(top_stagec_importance["feature"], top_stagec_importance["importance"])
    plt.title(f"Stage C Best Model Feature Importance ({stagec_best_model_name})")
    plt.xlabel("Importance")
    stagec_importance_plot_path = save_current_figure("stageC_best_model_feature_importance.png")
else:
    stagec_best_importance = pd.DataFrame()
    STAGEC_BEST_FEATURE_IMPORTANCE_PATH = None
    stagec_importance_plot_path = None


def build_stagec_or_baseline_pipeline(label: str):
    if label == "stageB_reference":
        return make_pipeline(make_lgbm())
    return make_stagec_pipeline(stagec_best_model_name, stagec_best_clean_params)


def time_aware_learning_curve_for_model(label: str, features: list[str]) -> pd.DataFrame:
    validation_start = int(len(X_train) * 0.80)
    X_curve_train = X_train.iloc[:validation_start][features]
    y_curve_train = y_train.iloc[:validation_start]
    X_curve_valid = X_train.iloc[validation_start:][features]
    y_curve_valid = y_train.iloc[validation_start:]
    rows = []
    for fraction in np.linspace(0.30, 1.00, 6):
        n_rows = max(12, int(len(X_curve_train) * fraction))
        pipeline = build_stagec_or_baseline_pipeline(label)
        start_time = time.perf_counter()
        pipeline.fit(X_curve_train.iloc[:n_rows], y_curve_train.iloc[:n_rows])
        train_time = time.perf_counter() - start_time
        train_predictions = pipeline.predict(X_curve_train.iloc[:n_rows])
        valid_predictions = pipeline.predict(X_curve_valid)
        rows.append(
            {
                "model_label": label,
                "training_rows": n_rows,
                "train_rmse": math.sqrt(mean_squared_error(y_curve_train.iloc[:n_rows], train_predictions)),
                "validation_rmse": math.sqrt(mean_squared_error(y_curve_valid, valid_predictions)),
                "training_time_seconds": train_time,
            }
        )
    return pd.DataFrame(rows)

stagec_learning_curve = pd.concat(
    [
        time_aware_learning_curve_for_model("stageB_reference", stageb_reference_features),
        time_aware_learning_curve_for_model("stageC_tuned", stagec_best_features),
    ],
    ignore_index=True,
)
stagec_learning_curve.to_csv(STAGEC_LEARNING_CURVE_PATH, index=False)

plt.figure(figsize=(9, 5))
for label, group in stagec_learning_curve.groupby("model_label"):
    plt.plot(group["training_rows"], group["validation_rmse"], marker="o", label=f"{label} validation")
plt.title("Stage C Time-Aware Learning Curve")
plt.xlabel("Training rows")
plt.ylabel("Validation RMSE")
plt.legend()
plt.grid(alpha=0.25)
stagec_learning_curve_plot_path = save_current_figure("stageC_learning_curves.png")

stagec_residual_diagnostics = {
    "stageB_reference_lag1_residual_autocorr": float(pd.Series(stageb_reference_residuals).autocorr(lag=1)),
    "stageC_tuned_lag1_residual_autocorr": float(pd.Series(stagec_best_residuals).autocorr(lag=1)),
    "stageB_reference_time_residual_corr": float(np.corrcoef(np.arange(len(stageb_reference_residuals)), stageb_reference_residuals)[0, 1]),
    "stageC_tuned_time_residual_corr": float(np.corrcoef(np.arange(len(stagec_best_residuals)), stagec_best_residuals)[0, 1]),
}

display(stagec_best_prediction_frame.head())
display(stagec_improvement_summary)
display(stagec_residual_diagnostics)


## 17. Stage C Report

The report summarizes feature-ranking differences, search results, the strongest hyperparameter effects, overfitting behavior, residual diagnostics, and whether the tuned model is practically and statistically better than the Stage B reference.


In [ ]:
best_tuned_row = stagec_model_comparison.iloc[0]
stageb_row = stagec_model_comparison[stagec_model_comparison["version"] == "StageB_reference"].iloc[0]

rmse_improvement = float(stageb_row["test_rmse"] - best_tuned_row["test_rmse"])
mae_improvement = float(stageb_row["test_mae"] - best_tuned_row["test_mae"])
r2_improvement = float(best_tuned_row["test_r2"] - stageb_row["test_r2"])
rmse_pct_improvement = rmse_improvement / float(stageb_row["test_rmse"]) if float(stageb_row["test_rmse"]) else np.nan
mae_pct_improvement = mae_improvement / float(stageb_row["test_mae"]) if float(stageb_row["test_mae"]) else np.nan

stageb_gap = float(stageb_row["test_rmse"] - stageb_row["cv_rmse"])
stagec_gap = float(best_tuned_row["test_rmse"] - best_tuned_row["cv_rmse"])
overfitting_note = (
    "The tuned model reduced the CV-to-test RMSE gap, which suggests less overfitting or better robustness on the latest period."
    if abs(stagec_gap) < abs(stageb_gap)
    else "The tuned model did not reduce the CV-to-test RMSE gap; the latest period remains harder than the validation windows."
)

baseline_resid_autocorr = abs(stagec_residual_diagnostics["stageB_reference_lag1_residual_autocorr"])
tuned_resid_autocorr = abs(stagec_residual_diagnostics["stageC_tuned_lag1_residual_autocorr"])
residual_randomness_note = (
    "The tuned model has lower absolute lag-1 residual autocorrelation, so its residuals are more random over time by this diagnostic."
    if tuned_resid_autocorr < baseline_resid_autocorr
    else "The tuned model does not reduce lag-1 residual autocorrelation, so residual time structure still remains."
)

rmse_bootstrap = stagec_bootstrap_improvement[stagec_bootstrap_improvement["metric"] == "rmse"].iloc[0]
mae_bootstrap = stagec_bootstrap_improvement[stagec_bootstrap_improvement["metric"] == "mae"].iloc[0]
r2_bootstrap = stagec_bootstrap_improvement[stagec_bootstrap_improvement["metric"] == "r2"].iloc[0]
statistical_note = (
    "The paired bootstrap interval for RMSE improvement is fully above zero, so the Stage C RMSE gain has statistical support on the held-out period."
    if float(rmse_bootstrap["ci95_lower"]) > 0
    else "The paired bootstrap interval for RMSE improvement crosses zero, so the Stage C RMSE gain is not statistically conclusive on the held-out period."
)
practical_note = (
    "The RMSE or MAE improvement is at least 5%, so the tuned model is practically meaningfully better by this threshold."
    if (rmse_pct_improvement >= 0.05 or mae_pct_improvement >= 0.05)
    else "The RMSE and MAE improvements are below 5%, so any practical improvement should be treated as modest."
)

top_effects_text = stagec_hyperparameter_effects.head(10).to_string(index=False, float_format=lambda value: f"{value:.4f}")
comparison_text = stagec_model_comparison.drop(columns=["best_params"], errors="ignore").to_string(index=False, float_format=lambda value: f"{value:.4f}")
feature_selection_text = stagec_feature_selection_comparison.head(10).to_string(index=False, float_format=lambda value: f"{value:.4f}")
bootstrap_text = stagec_bootstrap_improvement.to_string(index=False, float_format=lambda value: f"{value:.4f}")

stagec_report = f"""# MmarakaAI Stage C Hyperparameter Optimization Report

## Objective

Stage C extended the Stage B classical ML notebook without changing preprocessing, feature engineering, chronological train/test splitting, or the original Random Forest feature-selection baseline. The fixed Stage B reference is LightGBMRegressor using the top 100 Random-Forest-ranked features.

## Feature-selection comparison

Stage C added LightGBM feature importance as a second ranking method and compared Random Forest and LightGBM rankings across these top-N subset sizes: {STAGEC_TOP_N_OPTIONS}.

The best feature set chosen for LightGBM was `{stagec_model_feature_choices['LightGBMRegressor']['feature_subset']}` using `{stagec_model_feature_choices['LightGBMRegressor']['ranking_method']}`.

The best feature set chosen for Random Forest was `{stagec_model_feature_choices['RandomForestRegressor']['feature_subset']}` using `{stagec_model_feature_choices['RandomForestRegressor']['ranking_method']}`.

Top feature-selection runs:

```text
{feature_selection_text}
```

## Hyperparameter optimization result

Best tuned model: **{stagec_best_model_name}**

Best search stage: **{stagec_best_search_stage}**

Selected feature ranking: **{stagec_best_feature_info['ranking_method']}**

Selected feature subset: **{stagec_best_feature_info['feature_subset']}**

Best hyperparameters:

```json
{json.dumps(json_safe(stagec_best_clean_params), indent=2)}
```

## Baseline versus tuned model

```text
{comparison_text}
```

Held-out test improvements relative to Stage B reference:

- RMSE delta: {rmse_improvement:.4f} ({rmse_pct_improvement:.2%})
- MAE delta: {mae_improvement:.4f} ({mae_pct_improvement:.2%})
- R-squared delta: {r2_improvement:.4f}

Positive RMSE and MAE deltas mean the tuned model reduced error. Positive R-squared delta means the tuned model explained more held-out variation.

## Hyperparameters with the greatest observed effect

The table below ranks hyperparameters by the range in mean cross-validated RMSE observed during the broad randomized search. A larger range means the model was more sensitive to that hyperparameter in this experiment.

```text
{top_effects_text}
```

## Overfitting check

Stage B CV-to-test RMSE gap: {stageb_gap:.4f}

Stage C CV-to-test RMSE gap: {stagec_gap:.4f}

{overfitting_note}

## Residual behavior

Stage B lag-1 residual autocorrelation: {stagec_residual_diagnostics['stageB_reference_lag1_residual_autocorr']:.4f}

Stage C lag-1 residual autocorrelation: {stagec_residual_diagnostics['stageC_tuned_lag1_residual_autocorr']:.4f}

Stage B residual-time correlation: {stagec_residual_diagnostics['stageB_reference_time_residual_corr']:.4f}

Stage C residual-time correlation: {stagec_residual_diagnostics['stageC_tuned_time_residual_corr']:.4f}

{residual_randomness_note}

## Statistical and practical comparison

Paired bootstrap improvements on the held-out test period:

```text
{bootstrap_text}
```

{statistical_note}

{practical_note}

## Outputs

- Best tuned model: `{STAGEC_MODEL_PATH}`
- Best hyperparameters: `{STAGEC_BEST_HYPERPARAMETERS_PATH}`
- Feature rankings: `{STAGEC_FEATURE_RANKINGS_PATH}`
- Feature-selection comparison: `{STAGEC_FEATURE_SELECTION_COMPARISON_PATH}`
- Hyperparameter search results: `{STAGEC_SEARCH_RESULTS_PATH}`
- Tuned model comparison: `{STAGEC_MODEL_COMPARISON_PATH}`
- Test predictions: `{STAGEC_TEST_PREDICTIONS_PATH}`
- Figures: `{FIGURE_DIR}`

## Conclusion

Stage C should be accepted over Stage B only if it improves cross-validation metrics and the held-out test period without increasing residual structure or overfitting. The ranked comparison table and bootstrap intervals above provide the evidence for that decision.
"""

STAGEC_REPORT_PATH.write_text(stagec_report, encoding="utf-8")
logger.info("Saved Stage C report to %s", STAGEC_REPORT_PATH)
print(f"Stage C report: {STAGEC_REPORT_PATH}")


## 18. Stage C Artifact Check and GitHub Sync

This final cell verifies Stage B and Stage C artifacts. In Colab, it pushes the generated artifacts back to GitHub using a `GITHUB_TOKEN` Colab secret with repository contents read/write permission. A normal Colab runtime cannot write directly to your laptop's local repository, so GitHub sync is the reliable bridge.


In [ ]:
import os
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path

IN_COLAB_RUNTIME = "google.colab" in sys.modules
GITHUB_REPO_SLUG = "Wiz-Tech-Kid/MmarakaAI"
GITHUB_BRANCH = "main"
AUTO_PUSH_ARTIFACTS_TO_GITHUB = True
ARTIFACT_ZIP_PATH = OUTPUT_DIR / "mmarakai_stageB_stageC_artifacts.zip"

stageb_artifacts = [
    MODEL_PATH,
    FEATURE_IMPORTANCE_PATH,
    MODEL_COMPARISON_PATH,
    SELECTED_FEATURES_PATH,
    FINAL_METRICS_PATH,
    REPORT_MD_PATH,
    REPORT_TXT_PATH,
]
stagec_artifacts = [
    STAGEC_MODEL_PATH,
    STAGEC_BEST_HYPERPARAMETERS_PATH,
    STAGEC_FEATURE_RANKINGS_PATH,
    STAGEC_SELECTED_FEATURES_PATH,
    STAGEC_FEATURE_SELECTION_COMPARISON_PATH,
    STAGEC_SEARCH_RESULTS_PATH,
    STAGEC_HYPERPARAMETER_EFFECTS_PATH,
    STAGEC_MODEL_COMPARISON_PATH,
    STAGEC_TEST_PREDICTIONS_PATH,
    STAGEC_LEARNING_CURVE_PATH,
    STAGEC_BOOTSTRAP_PATH,
    STAGEC_REPORT_PATH,
]
figure_artifacts = sorted(FIGURE_DIR.glob("stageB_*.png")) + sorted(FIGURE_DIR.glob("stageC_*.png"))
artifact_paths = [*stageb_artifacts, *stagec_artifacts, *figure_artifacts]

artifact_check = pd.DataFrame(
    [
        {
            "artifact": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else 0,
        }
        for path in artifact_paths
    ]
)
display(artifact_check)
missing_artifacts = artifact_check.loc[~artifact_check["exists"], "artifact"].tolist()
if missing_artifacts:
    raise FileNotFoundError("Missing expected artifacts: " + ", ".join(missing_artifacts))

with zipfile.ZipFile(ARTIFACT_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for artifact_path in artifact_paths:
        archive.write(artifact_path, artifact_path.relative_to(PROJECT_ROOT))
print(f"Artifact bundle: {ARTIFACT_ZIP_PATH}")


def run_git(args: list[str], check: bool = True, env: dict[str, str] | None = None) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=False,
        env=env,
    )
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip() or f"git {' '.join(args)} failed")
    return result


def get_colab_secret(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        return value
    if not IN_COLAB_RUNTIME:
        return None
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def push_artifacts_to_github() -> None:
    token = get_colab_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError(
            "Cannot push artifacts to GitHub because GITHUB_TOKEN is missing. "
            "In Colab, open the key icon, add a secret named GITHUB_TOKEN, "
            "and use a GitHub fine-grained token with Contents: Read and write access for this repository."
        )

    run_git(["config", "user.name", os.environ.get("GITHUB_USER_NAME", "MmarakaAI Colab")])
    run_git(["config", "user.email", os.environ.get("GITHUB_USER_EMAIL", "mmarakai-colab@users.noreply.github.com")])

    relative_paths = [str(path.relative_to(PROJECT_ROOT)) for path in artifact_paths if path.exists()]
    run_git(["add", *relative_paths])
    run_git(["add", "-f", str(MODEL_PATH.relative_to(PROJECT_ROOT))])
    run_git(["add", "-f", str(STAGEC_MODEL_PATH.relative_to(PROJECT_ROOT))])

    status = run_git(["status", "--short"], check=False).stdout.strip()
    if not status:
        print("No artifact changes to commit.")
        return

    print("Git changes that will be committed:")
    print(status)

    commit_result = run_git(["commit", "-m", "Add Stage B and Stage C ML artifacts"], check=False)
    commit_output = commit_result.stderr.strip() or commit_result.stdout.strip()
    if commit_result.returncode != 0 and "nothing to commit" not in commit_output.lower():
        raise RuntimeError(commit_output)

    with tempfile.TemporaryDirectory() as temp_dir:
        askpass_path = Path(temp_dir) / "git_askpass.py"
        askpass_path.write_text(
            "import os, sys\n"
            "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
            "print(os.environ['GITHUB_TOKEN'] if 'password' in prompt else 'x-access-token')\n",
            encoding="utf-8",
        )
        askpass_path.chmod(askpass_path.stat().st_mode | stat.S_IXUSR)
        push_env = os.environ.copy()
        push_env["GITHUB_TOKEN"] = token
        push_env["GIT_ASKPASS"] = str(askpass_path)
        push_env["GIT_TERMINAL_PROMPT"] = "0"
        push_result = subprocess.run(
            ["git", "push", f"https://github.com/{GITHUB_REPO_SLUG}.git", f"HEAD:{GITHUB_BRANCH}"],
            cwd=PROJECT_ROOT,
            text=True,
            capture_output=True,
            check=False,
            env=push_env,
        )
    if push_result.returncode != 0:
        raise RuntimeError(push_result.stderr.strip() or push_result.stdout.strip() or "git push failed")
    print(f"Artifacts pushed to GitHub: https://github.com/{GITHUB_REPO_SLUG}/tree/{GITHUB_BRANCH}")


if IN_COLAB_RUNTIME and AUTO_PUSH_ARTIFACTS_TO_GITHUB:
    push_artifacts_to_github()
elif IN_COLAB_RUNTIME:
    print("Colab artifacts were created but AUTO_PUSH_ARTIFACTS_TO_GITHUB is False.")
else:
    print("Running locally: artifacts were written directly into this local repository.")
